# Visualize Spectrum Reconstructions & Redshift Predictions

Load a trained transformer checkpoint (Approach A or B) plus the frozen spectrum tokenizer it was trained with, run it on a handful of DESI spectra, and visualize:

1. **Teacher-forced reconstruction** — predicted spectrum tokens decoded back through the tokenizer
2. **Autoregressive generation** — `model.generate()` from `[SOS]` only, no teacher forcing (the honest visualization)
3. **Redshift prediction** — predicted bin → decoded z value vs true z (scatter)
4. **Per-position token accuracy** — heatmap showing which positions in the sequence the model is good at
5. **Reconstruction error distribution** — per-spectrum MSE histogram

Set the paths in the **Configuration** cell, then run all cells top-to-bottom.

## Imports

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))

import json
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.models.transformer import (
    SpectrumTransformer,
    SOS_TOKEN, EOS_TOKEN,
    SPECTRUM_TOKEN_OFFSET, REDSHIFT_TOKEN_OFFSET, TOTAL_VOCAB_SIZE,
)
from src.tokenizers.spectrum import SpectrumTokenizer, N_TOKENS
from src.tokenizers.redshift import RedshiftTokenizer
from src.training.sequences import tokenize_and_build
from src.utils.data import DESISpectrumDataset

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.facecolor'] = 'white'

## Configuration

Point these at the artifacts you want to inspect.

### Using a NERSC-trained model locally

Copy the checkpoint + tokenizer + config from Perlmutter to your laptop:

```bash
# from your laptop
mkdir -p checkpoints/nersc/approach_a checkpoints/nersc/tokenizer_v1
scp perlmutter:/global/u1/j/joe2k/redshifty/checkpoints/nersc/approach_a_<JOBID>/best.pt \
    checkpoints/nersc/approach_a/best.pt
scp perlmutter:/global/u1/j/joe2k/redshifty/checkpoints/nersc/approach_a_<JOBID>/config.json \
    checkpoints/nersc/approach_a/
# tokenizer ckpt usually lives on $SCRATCH:
scp perlmutter:'$SCRATCH/deepsrch/checkpoints/tokenizer_v1_<JOBID>/best.pt' \
    checkpoints/nersc/tokenizer_v1/best.pt
```

Phase-10+ transformer checkpoints embed the `z_tokenizer` state directly in the `.pt`, so redshift bins decode correctly without re-fitting. If your checkpoint is older, set `MANIFEST_PATH` to a copy of the training manifest, or accept a warning about locally-fit bins being miscalibrated.

### Running on NERSC directly

If you're on a Perlmutter login node with Jupyter, just point the paths at `$SCRATCH` / `$CFS` and set `DATA_DIR=/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix/sv3/bright/100/10016` (or any single healpix dir).

### Settings

In [ ]:
# Edit these to point at your artifacts.
TRANSFORMER_CKPT = REPO / 'checkpoints/nersc/approach_a/best.pt'
TOKENIZER_CKPT   = REPO / 'checkpoints/nersc/tokenizer_v1/best.pt'
DATA_DIR         = REPO / 'data/desi_raw'
MANIFEST_PATH    = None        # optional: Path('...') to a training JSONL manifest
APPROACH         = None        # 'a' / 'b' / None to auto-detect from checkpoint
N_SAMPLES        = 6
SEED             = 42
ENCODER_MASK_RATIO = 0.0       # set >0 to mask encoder spectrum tokens during visualization

print(f'TRANSFORMER_CKPT = {TRANSFORMER_CKPT}')
print(f'TOKENIZER_CKPT   = {TOKENIZER_CKPT}')
print(f'DATA_DIR         = {DATA_DIR}')
print(f'MANIFEST_PATH    = {MANIFEST_PATH}')
print(f'APPROACH         = {APPROACH!r} (auto-detect if None)')
assert TRANSFORMER_CKPT.is_file(), f'Missing transformer ckpt: {TRANSFORMER_CKPT}'
assert TOKENIZER_CKPT.is_file(),   f'Missing tokenizer ckpt: {TOKENIZER_CKPT}'
assert DATA_DIR.is_dir(),          f'Missing data dir: {DATA_DIR}'

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'device = {device}')

## (Optional) List W&B runs & their model artifacts

Discover what's available before downloading. Lists every run in the project (most recent first) with state, best val loss, and any uploaded model artifacts (with versions and aliases). Copy any artifact URI from the printout into `ARTIFACT_URI` below.

Set `WANDB_ENTITY` to your wandb username; project is `redshifty` by default.

In [ ]:
WANDB_ENTITY = None    # e.g. 'jonathansamuel'; if None, uses your default entity
WANDB_PROJECT = 'redshifty'
MAX_RUNS = 20

try:
    from dotenv import load_dotenv
    load_dotenv(REPO / '.env')
except ImportError:
    pass
import wandb
api = wandb.Api()
path = f'{WANDB_ENTITY}/{WANDB_PROJECT}' if WANDB_ENTITY else WANDB_PROJECT
runs = api.runs(path, per_page=MAX_RUNS)
print(f'\n=== runs in {path} (most recent first) ===\n')
for r in list(runs)[:MAX_RUNS]:
    best = r.summary.get('val/loss') or r.summary.get('val_loss') or '-'
    print(f'{r.name:50s}  state={r.state:10s}  best_val_loss={best}')
    try:
        arts = list(r.logged_artifacts())
    except Exception as e:
        arts = []
        print(f'    (could not list artifacts: {e})')
    for a in arts:
        if a.type != 'model':
            continue
        uri = f'{a.entity}/{a.project}/{a.name}'
        aliases = ','.join(a.aliases) if a.aliases else ''
        size_mb = a.size / 1e6 if a.size else 0
        print(f'    -> {uri:60s}  aliases=[{aliases}]  {size_mb:.0f} MB')

print('\nCopy a `... /artifact_name:version` URI into ARTIFACT_URI below.')
print('Use :best, :latest, or :v0/:v1/... to pin a specific version.')

## (Optional) Pull checkpoints from W&B Artifacts

Skip the `scp` step entirely by pulling the trained transformer **and/or** the frozen spectrum tokenizer straight from W&B.

- **Transformer** — the Phase-10+ training loop uploads on every best-val step (see `--push-wandb-artifact`). Pick a URI from the run-listing cell above.
- **Spectrum tokenizer** — uploaded once with `nersc/push_tokenizer_artifact.py` (run on NERSC after pretraining finishes). Typical URI: `<entity>/redshifty/spectrum_tokenizer_v1:best`.

Leave a URI as `None` to use the locally-set path instead.

In [ ]:
ARTIFACT_URI           = None  # transformer; e.g. 'jonathansamuel/redshifty/approach_a_<run>:best'
TOKENIZER_ARTIFACT_URI = None  # tokenizer;   e.g. 'jonathansamuel/redshifty/spectrum_tokenizer_v1:best'

if ARTIFACT_URI is not None or TOKENIZER_ARTIFACT_URI is not None:
    try:
        from dotenv import load_dotenv
        load_dotenv(REPO / '.env')
    except ImportError:
        pass
    import wandb
    api = wandb.Api()

    def _pull(uri, label):
        art = api.artifact(uri, type='model')
        dl_dir = REPO / 'checkpoints' / 'wandb_artifacts' / uri.replace('/', '__').replace(':', '@')
        dl_dir.mkdir(parents=True, exist_ok=True)
        art.download(root=str(dl_dir))
        pts = sorted(dl_dir.glob('*.pt'))
        assert pts, f'No .pt file in artifact {uri}'
        print(f'[wandb] {label}: downloaded {uri} -> {pts[0]}')
        return pts[0]

    if ARTIFACT_URI is not None:
        TRANSFORMER_CKPT = _pull(ARTIFACT_URI, 'transformer')
    if TOKENIZER_ARTIFACT_URI is not None:
        TOKENIZER_CKPT = _pull(TOKENIZER_ARTIFACT_URI, 'tokenizer')
else:
    print('[wandb] both ARTIFACT_URI and TOKENIZER_ARTIFACT_URI are None; using local paths')

## Load the trained transformer

Reads model dims from `config.json` next to the checkpoint. Extracts the embedded `z_tokenizer` state and `approach` if present (Phase 10+ checkpoints).

In [ ]:
cfg_path = TRANSFORMER_CKPT.parent / 'config.json'
if cfg_path.is_file():
    cfg = json.loads(cfg_path.read_text())
    print(f'config: d_model={cfg.get("d_model")}, '
          f'enc={cfg.get("n_encoder_layers")}, dec={cfg.get("n_decoder_layers")}, '
          f'heads={cfg.get("n_heads")}')
else:
    print('No config.json next to checkpoint; using d_model=768, 6/6 layers, 12 heads.')
    cfg = {'d_model': 768, 'n_encoder_layers': 6, 'n_decoder_layers': 6, 'n_heads': 12, 'dropout': 0.1}

model = SpectrumTransformer(
    d_model=cfg.get('d_model', 768),
    n_encoder_layers=cfg.get('n_encoder_layers', 6),
    n_decoder_layers=cfg.get('n_decoder_layers', 6),
    n_heads=cfg.get('n_heads', 12),
    dropout=0.0,
).to(device)

ckpt = torch.load(TRANSFORMER_CKPT, map_location=device, weights_only=False)
sd = ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt
model.load_state_dict(sd)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

n_params = sum(p.numel() for p in model.parameters())
print(f'Transformer loaded: {n_params:,} params (~{n_params/1e6:.1f}M)')

# Auto-detect approach from checkpoint if not explicitly set.
if APPROACH is None:
    APPROACH = (ckpt.get('approach') if isinstance(ckpt, dict) else None) or cfg.get('approach') or 'a'
    print(f'  approach (auto-detected) = {APPROACH!r}')
else:
    print(f'  approach (manual) = {APPROACH!r}')

# Extract z_tokenizer state if embedded (Phase 10+).
ckpt_z_state = ckpt.get('z_tokenizer') if isinstance(ckpt, dict) else None
if ckpt_z_state is not None:
    print(f'  z_tokenizer state found in checkpoint '
          f'(n_levels={ckpt_z_state["n_levels"]}, '
          f'n_z_samples={len(ckpt_z_state["sorted_z"])})')
else:
    print('  no embedded z_tokenizer state -- will fall back to MANIFEST_PATH or local fit')

## Load the spectrum tokenizer

Frozen during transformer training. Used here both to tokenize raw spectra (encoder input) and to decode predicted tokens back into flux space.

In [ ]:
spec_tok = SpectrumTokenizer().to(device)
tok_ckpt = torch.load(TOKENIZER_CKPT, map_location=device, weights_only=False)
tok_sd = tok_ckpt.get('model', tok_ckpt) if isinstance(tok_ckpt, dict) else tok_ckpt
spec_tok.load_state_dict(tok_sd)
spec_tok.eval()
for p in spec_tok.parameters():
    p.requires_grad_(False)
print('Spectrum tokenizer loaded.')

## Initialize the redshift tokenizer

Priority order:
1. **Embedded state in transformer checkpoint** (Phase 10+). Exactly matches training bins.
2. **Manifest of training redshifts** (`MANIFEST_PATH`). Re-fits on the same z distribution.
3. **Local FITS files** (fallback). Warning printed — bins won't match training and `z_acc` numbers will be misleading.

In [ ]:
z_tok = None

# Priority 1: embedded state
if ckpt_z_state is not None:
    z_tok = RedshiftTokenizer(
        n_levels=int(ckpt_z_state['n_levels']),
        gaussian_range=float(ckpt_z_state.get('gaussian_range', 3.0)),
    )
    z_tok._sorted_z = ckpt_z_state['sorted_z'].clone()
    z_tok._min_z = z_tok._sorted_z[0].item()
    z_tok._max_z = z_tok._sorted_z[-1].item()
    print(f'[z_tok] loaded from checkpoint  n={len(z_tok._sorted_z)}  '
          f'range=[{z_tok._min_z:.4f}, {z_tok._max_z:.4f}]')

# Priority 2: manifest
elif MANIFEST_PATH is not None and Path(MANIFEST_PATH).is_file():
    from astropy.io import fits
    zs = []
    with open(MANIFEST_PATH) as f:
        for line in f:
            rec = json.loads(line)
            try:
                with fits.open(rec['redrock'], memmap=True) as h:
                    z = h['REDSHIFTS'].data['Z']
                    zwarn = h['REDSHIFTS'].data['ZWARN']
                    zs.append(z[zwarn == 0].astype('float32'))
            except Exception as e:
                print(f'  skip {rec["redrock"]}: {e}')
    zs = torch.from_numpy(np.concatenate(zs))
    z_tok = RedshiftTokenizer(n_levels=256)
    z_tok.fit(zs)
    print(f'[z_tok] fit from manifest  n={len(zs)}  '
          f'range=[{zs.min():.4f}, {zs.max():.4f}]')

## Load spectra to visualize

Reuses `DESISpectrumDataset` from `src/utils/data.py` to load FITS files from `DATA_DIR`. If `z_tok` is still uninitialized at this point, fits it on these spectra as a last resort (and prints a warning).

In [ ]:
coadd_files = sorted(DATA_DIR.glob('coadd-*.fits'))
print(f'Found {len(coadd_files)} coadd files in {DATA_DIR}')

spectra = []
for f in coadd_files:
    rr = f.parent / f.name.replace('coadd-', 'redrock-')
    if not rr.exists():
        print(f'  skip {f.name}: no redrock counterpart')
        continue
    ds = DESISpectrumDataset(
        coadd_path=f, redrock_path=rr,
        require_good_zwarn=True, require_nonzero_flux=True,
    )
    for i in range(len(ds)):
        spectra.append(ds[i])
print(f'Loaded {len(spectra)} good spectra')

if z_tok is None:
    print('\n*** WARNING: z_tok was not loaded from checkpoint or manifest. ***')
    print('*** Fitting on local FITS spectra. Predicted z values may be miscalibrated ***')
    print('*** relative to what the trained model expects. ***\n')
    z_values_fallback = torch.tensor([float(s['z']) for s in spectra])
    z_tok = RedshiftTokenizer(n_levels=256)
    z_tok.fit(z_values_fallback)
    print(f'[z_tok] fit from local FITS  n={len(z_values_fallback)}')

print(f'\n[z_tok] in use: n_levels={z_tok.n_levels}  '
      f'range=[{z_tok._min_z:.4f}, {z_tok._max_z:.4f}]')

## Pick samples to visualize

In [ ]:
rng = torch.Generator().manual_seed(SEED)
idx = torch.randperm(len(spectra), generator=rng)[:N_SAMPLES].tolist()
sample_specs = [spectra[i] for i in idx]
for i, s in enumerate(sample_specs):
    print(f'  sample {i}: z={float(s["z"]):+.4f}  L={len(s["flux"])} pixels')

## Helpers

In [ ]:
def specs_to_batch(spec_list):
    """Pad a list of spectrum dicts to a uniform tensor batch."""
    L = max(s['flux'].shape[0] for s in spec_list)
    def _pad(t, fill=0.0):
        if t.shape[0] == L: return t
        out = torch.full((L,), fill, dtype=t.dtype)
        out[:t.shape[0]] = t
        return out
    return {
        'flux': torch.stack([_pad(s['flux']) for s in spec_list]),
        'ivar': torch.stack([_pad(s['ivar'], 0.0) for s in spec_list]),
        'z':    torch.stack([s['z'] for s in spec_list]).float(),
    }

@torch.no_grad()
def predict_teacher_forced(batch, approach):
    """Returns (enc, dec, target, predicted_tokens, denorm). predicted_tokens
    has the same shape as target — model's argmax at every decoder position."""
    enc, dec, tgt, _ = tokenize_and_build(
        batch, spec_tok, z_tok, approach, device, encoder_mask_ratio=ENCODER_MASK_RATIO,
    )
    logits, _ = model(enc, dec, targets=tgt)
    pred = logits.argmax(dim=-1)
    flux = batch['flux'].to(device)
    ivar = batch['ivar'].to(device)
    istd = torch.sqrt(ivar.clamp(min=1e-10))
    x = torch.stack([flux, istd], dim=1)
    _, denorm = spec_tok.encode(x)
    return enc, dec, tgt, pred, denorm

@torch.no_grad()
def predict_autoregressive(batch, approach, temperature=1.0):
    enc, _, tgt, _ = tokenize_and_build(
        batch, spec_tok, z_tok, approach, device, encoder_mask_ratio=ENCODER_MASK_RATIO,
    )
    L_dec = tgt.shape[1]
    generated = model.generate(
        enc, decoder_start_token=SOS_TOKEN, max_new_tokens=L_dec,
        temperature=temperature,
    )
    return generated, tgt

def decode_to_flux(spec_token_ids, denorm):
    """Map global vocab token IDs (with SPECTRUM_TOKEN_OFFSET) back through
    the spectrum tokenizer to a continuous flux array. Invalid token IDs
    (SOS/EOS/MASK leakage) are mapped to LFQ index 0 so decode doesn't crash."""
    lfq_idx = spec_token_ids - SPECTRUM_TOKEN_OFFSET
    valid = (lfq_idx >= 0) & (lfq_idx < 1024)
    lfq_idx = torch.where(valid, lfq_idx, torch.zeros_like(lfq_idx))
    decoded = spec_tok.decode(lfq_idx, denorm)
    return decoded[:, 0]  # flux channel

## Visualization 1 — Teacher-forced reconstruction

Each row: one sample. Black = original (after tokenizer roundtrip); orange = decoded from the model's predicted tokens. Teacher-forced means the decoder saw the ground-truth tokens at every position before the one it predicted, so this is the **best-case** reconstruction (and the most copy-inflated).

In [ ]:
batch = specs_to_batch(sample_specs)
enc, dec, tgt, pred, denorm = predict_teacher_forced(batch, APPROACH)

pred_spec_tokens = pred[:, 1:1 + N_TOKENS]
true_spec_tokens = tgt[:, 1:1 + N_TOKENS]

pred_flux = decode_to_flux(pred_spec_tokens, denorm).cpu().numpy()
true_flux = decode_to_flux(true_spec_tokens, denorm).cpu().numpy()

fig, axes = plt.subplots(N_SAMPLES, 1, figsize=(11, 1.8 * N_SAMPLES), sharex=True)
if N_SAMPLES == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.plot(true_flux[i], color='black', lw=0.6, label='true (tokenizer roundtrip)')
    ax.plot(pred_flux[i], color='C1', lw=0.6, alpha=0.85, label='predicted (teacher-forced)')
    z = float(sample_specs[i]['z'])
    ax.set_ylabel(f'z={z:+.3f}', fontsize=9)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('latent-grid position (8704 pixels)')
fig.suptitle(f'Approach {APPROACH.upper()} — teacher-forced reconstructions ({N_SAMPLES} samples)')
fig.tight_layout()
plt.show()

## Visualization 2 — Autoregressive generation

Same samples, but generated token-by-token starting from `[SOS]` only. **No peeking at the answer.** This is the honest reconstruction — if it looks similar to teacher-forced, the model really learned. If it diverges, the teacher-forced number was inflated by the cross-attention copy.

In [ ]:
generated, tgt_ar = predict_autoregressive(batch, APPROACH, temperature=1.0)
ar_spec_tokens = generated[:, 2:2 + N_TOKENS]
# Pad if generate stopped early on EOS
if ar_spec_tokens.shape[1] < N_TOKENS:
    pad_len = N_TOKENS - ar_spec_tokens.shape[1]
    pad = torch.full((ar_spec_tokens.shape[0], pad_len), SPECTRUM_TOKEN_OFFSET,
                     dtype=torch.long, device=device)
    ar_spec_tokens = torch.cat([ar_spec_tokens, pad], dim=1)

ar_flux = decode_to_flux(ar_spec_tokens, denorm).cpu().numpy()

fig, axes = plt.subplots(N_SAMPLES, 1, figsize=(11, 1.8 * N_SAMPLES), sharex=True)
if N_SAMPLES == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.plot(true_flux[i], color='black', lw=0.6, label='true')
    ax.plot(ar_flux[i], color='C3', lw=0.6, alpha=0.85, label='autoregressive')
    z = float(sample_specs[i]['z'])
    ax.set_ylabel(f'z={z:+.3f}', fontsize=9)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('latent-grid position (8704 pixels)')
fig.suptitle(f'Approach {APPROACH.upper()} — autoregressive reconstructions ({N_SAMPLES} samples)')
fig.tight_layout()
plt.show()

## Visualization 3 — Redshift predictions

Predicted bin → decoded z (via the empirical-CDF inverse). For Approach A this should be near-perfect (cross-attention copy); for B it should be near random (the Phase 9 finding).

In [ ]:
tf_z_token  = pred[:, 0].cpu()
tf_z_idx    = (tf_z_token - REDSHIFT_TOKEN_OFFSET).clamp(0, z_tok.n_levels - 1)
tf_z_pred   = torch.tensor([z_tok.decode(int(i)) for i in tf_z_idx])

ar_z_token  = generated[:, 1].cpu()
ar_z_idx    = (ar_z_token - REDSHIFT_TOKEN_OFFSET).clamp(0, z_tok.n_levels - 1)
ar_z_pred   = torch.tensor([z_tok.decode(int(i)) for i in ar_z_idx])

z_true = torch.stack([s['z'] for s in sample_specs]).float()

fig, ax = plt.subplots(figsize=(6, 6))
lim_lo = min(z_true.min().item(), tf_z_pred.min().item(), ar_z_pred.min().item()) - 0.05
lim_hi = max(z_true.max().item(), tf_z_pred.max().item(), ar_z_pred.max().item()) + 0.05
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=0.7, alpha=0.5, label='y=x')
ax.scatter(z_true, tf_z_pred, s=70, alpha=0.75, label='teacher-forced', c='C1')
ax.scatter(z_true, ar_z_pred, s=70, alpha=0.75, marker='x', label='autoregressive', c='C3')
for i in range(len(z_true)):
    ax.annotate(str(i), (z_true[i], tf_z_pred[i]), fontsize=7, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('true z')
ax.set_ylabel('predicted z')
ax.set_title(f'Approach {APPROACH.upper()} — redshift prediction')
ax.grid(alpha=0.3)
ax.legend()
ax.set_xlim(lim_lo, lim_hi)
ax.set_ylim(lim_lo, lim_hi)
plt.show()

print(f'\nteacher-forced  MAE = {(tf_z_pred - z_true).abs().mean():.4f}')
print(f'autoregressive  MAE = {(ar_z_pred - z_true).abs().mean():.4f}')
print(f'\nPer-sample:')
for i in range(len(z_true)):
    print(f'  {i}: z_true={z_true[i]:+.4f}  tf={tf_z_pred[i]:+.4f}  ar={ar_z_pred[i]:+.4f}')

## Visualization 4 — Per-position token accuracy

Across the batch, fraction of correct token predictions at each decoder position. Position 0 = redshift; positions 1..N = spectrum tokens. A model relying on the trivial copy shortcut will be near-flat at high accuracy on positions 1+ but near-zero at position 0.

In [ ]:
tf_correct = (pred == tgt).float().mean(dim=0).cpu().numpy()
ar_overlap_len = min(generated.shape[1] - 1, tgt.shape[1])
ar_correct = (generated[:, 1:1 + ar_overlap_len] == tgt[:, :ar_overlap_len]).float().mean(dim=0).cpu().numpy()

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(tf_correct, label='teacher-forced', color='C1', lw=1.0)
ax.plot(ar_correct, label='autoregressive', color='C3', lw=1.0, alpha=0.8)
ax.axvspan(0, 0.5, alpha=0.15, color='C0', label='redshift token (pos 0)')
ax.set_xlabel('decoder position')
ax.set_ylabel('fraction correct across batch')
ax.set_title(f'Approach {APPROACH.upper()} — per-position prediction accuracy')
ax.set_ylim(-0.02, 1.05)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Visualization 5 — Reconstruction error distribution

Per-spectrum MSE between original (tokenizer-roundtripped) and predicted flux. The AR/TF ratio is the headline — `~1.0` means the model really learned; `>>1` means teacher-forcing was carrying it.

In [ ]:
tf_mse = ((torch.from_numpy(true_flux) - torch.from_numpy(pred_flux))**2).mean(dim=1).numpy()
ar_mse = ((torch.from_numpy(true_flux) - torch.from_numpy(ar_flux))**2).mean(dim=1).numpy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tf_mse, bins=20, alpha=0.6, label=f'teacher-forced (mean={tf_mse.mean():.3f})', color='C1')
ax.hist(ar_mse, bins=20, alpha=0.6, label=f'autoregressive  (mean={ar_mse.mean():.3f})', color='C3')
ax.set_xlabel('per-spectrum MSE')
ax.set_ylabel('count')
ax.set_title('Reconstruction error distribution')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print(f'TF MSE: mean={tf_mse.mean():.3f}, median={np.median(tf_mse):.3f}, max={tf_mse.max():.3f}')
print(f'AR MSE: mean={ar_mse.mean():.3f}, median={np.median(ar_mse):.3f}, max={ar_mse.max():.3f}')
print(f'AR/TF ratio: {ar_mse.mean()/max(tf_mse.mean(), 1e-9):.2f}x  '
      f'(1.0 = identical; >>1 = AR much worse → copy shortcut)')

## What to look for

| signal | interpretation |
|---|---|
| TF and AR reconstructions look similar | model has actually learned spectroscopy |
| TF looks great, AR looks like noise | the teacher-forced number was inflated by the cross-attention copy — the model can't actually generate |
| Redshift scatter on `y=x` for Approach A | cross-attention copy works (expected) |
| Redshift scatter scattered for Approach B | the encoder isn't encoding redshift (the Phase 9 result) |
| Per-position accuracy flat-high for pos 1+, low at pos 0 | classic copy regime — needs encoder masking |
| Per-position accuracy declining toward the end of the sequence | error accumulation in AR, typical of language models |
| AR/TF MSE ratio = 1.0 | honest model |
| AR/TF MSE ratio >>1 (e.g. 10x+) | strong copy dependency |